In [ ]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel

# --- 1. Point these at the function you're currently working on ---
INPUTS_PATH = "initial_inputs.npy"
OUTPUTS_PATH = "initial_outputs.npy"

X = np.load(INPUTS_PATH)
Y = np.load(OUTPUTS_PATH)

n_points, n_dims = X.shape
print(f"Loaded {n_points} points, {n_dims}-dimensional input")
print(f"Y range: [{Y.min():.4g}, {Y.max():.4g}]")

# --- 2. Fit a GP surrogate to what we've observed so far ---
kernel = RBF(length_scale=0.2, length_scale_bounds=(1e-2, 1e1)) + WhiteKernel(
    noise_level=1e-6, noise_level_bounds=(1e-10, 1e-1)
)
gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=8)
gp.fit(X, Y)

# --- 3. Build a candidate grid over [0, 1]^n_dims ---
# For higher dimensions, a dense grid explodes in size, so we use random
# candidates instead once n_dims gets large.
np.random.seed(0)
if n_dims <= 3:
    grid_res = 25 if n_dims == 3 else 60
    axes = [np.linspace(0, 1, grid_res) for _ in range(n_dims)]
    mesh = np.meshgrid(*axes)
    candidates = np.column_stack([m.ravel() for m in mesh])
else:
    n_candidates = 20000
    candidates = np.random.uniform(0, 1, size=(n_candidates, n_dims))

# --- 4. Predict mean + uncertainty at every candidate ---
post_mean, post_std = gp.predict(candidates, return_std=True)

# --- 5. UCB acquisition function (balance exploration/exploitation) ---
beta = 1.96
acquisition = post_mean + beta * post_std

# --- 6. Pick the best candidate ---
best_idx = np.argmax(acquisition)
next_x = candidates[best_idx]

# --- 7. Format exactly as the portal requires: 0.xxxxxx-0.xxxxxx-... ---
formatted = "-".join(f"{v:.6f}" for v in next_x)

print("\nSuggested next query point:")
print(next_x)
print("\nPaste this into the portal:")
print(formatted)